## AGN Homework 1 – SDSS Database

The **Sloan Digital Sky Survey (SDSS)** is a cornerstone of modern observational cosmology and extragalactic astrophysics, providing the largest and most detailed threedimensional maps of the universe. It is indispensable in the study of **Active Galactic Nuclei (AGN)** because its widefield multifilter imaging and extensive spectroscopic followup have enabled the discovery and systematic classification of hundreds of thousands of quasars across a vast range of redshifts. Its database allows researchers to perform complex `SQL` queries to correlate multiwavelength photometric data with spectroscopic line measurements. This infrastructure supports the statistical study of the main sequence of active galactic cores, the evolution of supermassive black holes, and the relationship between central engines and their host galaxies.

In [1]:
import pandas as pd
import numpy as np
import matplotlib as mpl

import matplotlib.pyplot as plt

### [a)](https://skyserver.sdss.org/dr19/SearchTools/sql)

#### Task:

- select galaxies & quasars
- filter redshifts *0.05 < z < 0.3*
- ensure *SNR > 35* near *Hβ* line (band *g λ4000-5500* range)
- emission lines present
  - *[O III] λ5007*
  - *Hβ λ4863*
  - *Hγ λ4341*
- require *FWHM > 1000 km/s* for *Hβ*
- find flux ratios
  - *[O III] / Hβ*
  - *Hβ / Hγ*
  - *[O III] / Hγ*
- for *Hβ* search
  - equivalent width
  - flux
  - redshift
  - type *SFD* extinction correction *E(B-V)*

#### Query:

```sql
SELECT 
    (g.oiii_5007_flux / g.h_beta_flux) AS oiii_hbeta_ratio,
    (g.h_beta_flux / g.h_gamma_flux) AS hbeta_hgamma_ratio,
    (g.oiii_5007_flux / g.h_gamma_flux) AS oiii_hgamma_ratio,
    g.h_beta_eqw, 
    g.h_beta_flux, 
    s.z, 
    i.e_bv_sfd -- E(B-V) SFD extinction correction
FROM SpecObj AS s
JOIN galSpecLine AS g ON s.specobjid = g.specobjid
JOIN galSpecInfo AS i ON s.specobjid = i.specobjid
WHERE (s.class = 'QSO' OR s.class = 'GALAXY') 
  AND s.z BETWEEN 0.05 AND 0.3
  AND s.snmedian_g > 35
  AND g.oiii_5007_eqw < 0 
  AND g.h_beta_eqw < 0 
  AND g.h_gamma_eqw < 0
  AND g.h_beta_flux > 0
  AND g.h_gamma_flux > 0
  AND (2.355 * g.sigma_balmer) > 1000
```

#### Result:

In [5]:
df_a = pd.read_csv('a.csv', skiprows=1)
df_a

,oiii_hbeta_ratio,hbeta_hgamma_ratio,oiii_hgamma_ratio,h_beta_eqw,h_beta_flux,z,e_bv_sfd
0,0.813190,2.092379,1.701502,-24.484470,4723.809000,0.117997,0.049188
1,0.170785,2.185541,0.373259,-16.616000,2577.094000,0.296773,0.049731
2,2.810742,2.034963,5.719755,-25.179620,2802.729000,0.063291,0.027591
3,1.093161,1.824435,1.994401,-11.709680,3590.220000,0.063550,0.026815
4,0.243450,2.117563,0.515521,-14.253350,1618.641000,0.261068,0.025826
...,...,...,...,...,...,...,...
220,1.758161,0.652581,1.147343,-0.201658,9.687269,0.067036,0.029346
221,0.463524,1.779278,0.824737,-16.605150,2236.087000,0.127734,0.039514
222,0.414769,2.267872,0.940643,-15.904240,1450.228000,0.210923,0.047597
223,0.120378,1.060523,0.127664,-3.341009,694.192400,0.152479,0.013393


### [b)](https://skyserver.sdss.org/dr19/SearchTools/sql)

#### Task:

- count found objects
- determine which condition causes most severe narrowing

#### Query:

```sql
SELECT 
    count(*)
FROM SpecObj AS s
JOIN galSpecLine AS g ON s.specobjid = g.specobjid
JOIN galSpecInfo AS i ON s.specobjid = i.specobjid
WHERE (s.class = 'QSO' OR s.class = 'GALAXY') 
  AND s.z BETWEEN 0.05 AND 0.3
  AND s.snmedian_g > 35
  AND g.oiii_5007_eqw < 0 
  AND g.h_beta_eqw < 0 
  AND g.h_gamma_eqw < 0
  AND g.h_beta_flux > 0
  AND g.h_gamma_flux > 0
  AND (2.355 * g.sigma_balmer) > 1000
```

#### Method:

- sequentially comment out conditions via `--`

#### Result:

- *225* with all filters
- *405 (Δ180)* without `fwhm(g.sigma_balmer) > 1000`
- *227 (Δ2)* without `g.h_gamma_flux > 0`
- *229 (Δ4)* without `g.h_beta_flux > 0`
- *225 (Δ0)* without `g.h_gamma_eqw < 0`
- *225 (Δ0)* without `g.h_beta_eqw < 0`
- *227 (Δ2)* without `g.oiii_5007_eqw < 0 `
- *22350 (Δ22125)* without `s.snmedian_g > 35`
- *1436 (Δ1211)* without `0.05 < s.z < 0.3`

#### Conclusion:

- our *SNR* constraint is the most selective out of all conditions

### [c)](https://skyserver.sdss.org/dr19/SearchTools/sql)

#### Task:

- check for entires with subclass of active galactic nuclei

#### Query:

```sql
SELECT 
    (g.oiii_5007_flux / g.h_beta_flux) AS oiii_hbeta_ratio,
    (g.h_beta_flux / g.h_gamma_flux) AS hbeta_hgamma_ratio,
    (g.oiii_5007_flux / g.h_gamma_flux) AS oiii_hgamma_ratio,
    g.h_beta_eqw, 
    g.h_beta_flux, 
    s.z, 
    i.e_bv_sfd -- E(B-V) SFD extinction correction
FROM SpecObj AS s
JOIN galSpecLine AS g ON s.specobjid = g.specobjid
JOIN galSpecInfo AS i ON s.specobjid = i.specobjid
WHERE s.subclass = 'AGN' 
  AND s.z BETWEEN 0.05 AND 0.3
  AND s.snmedian_g > 35
  AND g.oiii_5007_eqw < 0 
  AND g.h_beta_eqw < 0 
  AND g.h_gamma_eqw < 0
  AND g.h_beta_flux > 0
  AND g.h_gamma_flux > 0
  AND (2.355 * g.sigma_balmer) > 1000
```

#### Result:

- none found

### [d)](https://skyserver.sdss.org/dr18/CrossMatchTools/ObjectCrossID) 

#### Task:

#### Query:

```sql
SELECT 
    u.plate, u.mjd, u.fiberid, -- uploaded list fields
    (g.oiii_5007_flux / g.h_beta_flux) AS oiii_hbeta_ratio,
    g.h_beta_eqw, 
    g.h_beta_flux, 
    s.z
FROM #upload AS u -- 287-plate-mjd-fiber.txt
JOIN SpecObj AS s ON s.plate = u.plate AND s.mjd = u.mjd AND s.fiberid = u.fiberid
JOIN galSpecLine AS g ON s.specobjid = g.specobjid
JOIN galSpecInfo AS i ON s.specobjid = i.specobjid
WHERE (s.class = 'QSO' OR s.class = 'GALAXY') 
  AND s.z BETWEEN 0.05 AND 0.6
  AND s.snmedian_g > 35
  AND g.oiii_5007_eqw < 0 
  AND g.h_beta_eqw < 0 
  AND g.h_gamma_eqw < 0
  AND g.h_beta_flux > 0
  AND g.h_gamma_flux > 0
  AND (2.355 * g.sigma_balmer) > 1000
```